# 02 Depth Normalization Tuning

Use this notebook after B1 has produced at least one preprocessed bundle. It plots raw depth statistics, checks hole filling, and writes visual comparisons for B2 acceptance.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from blast_pile_diffusion.data.sample_bundle import SampleBundle
from blast_pile_diffusion.preprocessing.depth_processor import fill_depth_holes, process_depth

PREPROCESSED_DIR = Path('../data/preprocessed')
OUTPUT_DIR = Path('../notebooks/outputs/depth_normalization')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bundle_dirs = sorted(p for p in PREPROCESSED_DIR.iterdir() if (p / 'rgb.png').exists())
assert bundle_dirs, f'No bundles found under {PREPROCESSED_DIR}. Run scripts/01_preprocess_unity.py first.'
bundle = SampleBundle.load(bundle_dirs[0])
depth = bundle.depth.astype(np.float32)
depth_filled = fill_depth_holes(depth)
depth_cn = process_depth(depth)
bundle.sample_key, depth.shape, depth_cn.shape, depth_cn.dtype


In [ ]:
valid = depth[np.isfinite(depth) & (depth > 0)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(valid.ravel(), bins=80)
axes[0].set_title('Raw depth histogram')
axes[0].set_xlabel('meters')
axes[1].imshow(depth, cmap='magma')
axes[1].set_title('Raw depth')
axes[1].axis('off')
axes[2].imshow(depth_cn)
axes[2].set_title('ControlNet depth (near bright)')
axes[2].axis('off')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{bundle.sample_key}_depth_summary.png', dpi=160)
plt.show()


In [ ]:
hole_mask = (~np.isfinite(depth)) | (depth <= 0)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(hole_mask, cmap='gray')
axes[0].set_title('Hole mask')
axes[1].imshow(depth, cmap='magma')
axes[1].set_title('Before fill')
axes[2].imshow(depth_filled, cmap='magma')
axes[2].set_title('After fill')
for ax in axes:
    ax.axis('off')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{bundle.sample_key}_hole_fill.png', dpi=160)
plt.show()
print({'holes': int(hole_mask.sum()), 'near_bright_check': bool(depth_cn[..., 0].max() > depth_cn[..., 0].min())})
